##### **IMPORT ALL THE ENVIRONMENTS**

In [33]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os

from dotenv import load_dotenv

from autogen_agentchat.agents import (
    AssistantAgent,
    CodeExecutorAgent
)

from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_agentchat.conditions import TextMentionTermination

from autogen_agentchat.ui import Console

from autogen_ext.code_executors.local import (
    LocalCommandLineCodeExecutor
)

from autogen_ext.models.openai import (
    OpenAIChatCompletionClient
)

In [34]:
# ============================================================
# 2. ENVIRONMENT SETUP
# ============================================================

load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. Add it to the project .env file."
    )

print("Gemini API key loaded:", bool(api_key))

Gemini API key loaded: True


In [35]:
# ============================================================
# 3. GEMINI MODEL CLIENT
# ============================================================

model_client = OpenAIChatCompletionClient(
    model="gemini-3.6-flash",
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
        "family": "gemini-3.6-flash",
    },
)

print("Gemini model client created successfully.")

Gemini model client created successfully.


In [36]:
# ============================================================
# 4. DIRECTORY AND DATASET SETUP
# ============================================================

project_root = os.getcwd()
if os.path.basename(project_root) == "research":
    project_root = os.path.abspath(os.path.join(project_root, ".."))

if not os.path.isdir(os.path.join(project_root, "data")):
    raise FileNotFoundError(f"Project root not found: {project_root}")

output_folder = os.path.join(project_root, "outputs")
coding_folder = os.path.join(project_root, "research", "coding")
os.makedirs(output_folder, exist_ok=True)
os.makedirs(coding_folder, exist_ok=True)
data_folder = os.path.join(project_root, "data")

print(f"Project root: {project_root}")
print(f"Data folder: {data_folder}")
print(f"Output folder: {output_folder}")
print(f"Executor folder: {coding_folder}")

Project root: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer
Data folder: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data
Output folder: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs
Executor folder: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\research\coding


In [37]:
# ============================================================
# LOCATE PROJECT ROOT AND DATASETS
# ============================================================

project_root = os.getcwd()
if os.path.basename(project_root) == "research":
    project_root = os.path.abspath(os.path.join(project_root, ".."))

if not os.path.isdir(os.path.join(project_root, "data")):
    raise FileNotFoundError(
        f"Project root not found. Expected a data folder under: {project_root}"
    )

output_folder = os.path.join(project_root, "outputs")
coding_folder = os.path.join(project_root, "coding")
os.makedirs(output_folder, exist_ok=True)
os.makedirs(coding_folder, exist_ok=True)

data_folder = os.path.join(project_root, "data")

for file in os.listdir(data_folder):
    print(file)

.gitkeep
bond_portfolio_data.csv
monte_carlo_scenarios.csv
yield_curve_history.csv


In [38]:
# ============================================================
# 4.1 SELECT DATASET
# ============================================================

csv_files = sorted(
    os.path.join(data_folder, filename)
    for filename in os.listdir(data_folder)
    if filename.lower().endswith(".csv")
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_folder}")

DATA_FILE = next(
    (path for path in csv_files if os.path.basename(path) == "bond_portfolio_data.csv"),
    csv_files[0],
)

print("Available CSV files:")
for path in csv_files:
    print(f"- {path}")
print(f"\nSelected dataset: {DATA_FILE}")

Available CSV files:
- c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data\bond_portfolio_data.csv
- c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data\monte_carlo_scenarios.csv
- c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data\yield_curve_history.csv

Selected dataset: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data\bond_portfolio_data.csv


In [39]:
# ============================================================
# 5. CODE EXECUTOR
# ============================================================

# The executor runs from research/coding. Paths inside executed code use
# ../../data and ../../outputs to reach the project root.
executor = LocalCommandLineCodeExecutor(work_dir=coding_folder)

code_executor = CodeExecutorAgent(
    name="Code_Executor",
    code_executor=executor,
)

print(f"Code executor created: {coding_folder}")

Code executor created: c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\coding


C:\Users\ADITHYA UBALE\AppData\Local\Temp\ipykernel_6620\4130634526.py:7: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(work_dir=coding_folder)
C:\Users\ADITHYA UBALE\AppData\Local\Temp\ipykernel_6620\4130634526.py:9: UserWarning: No approval function set for CodeExecutorAgent. This means code will be executed automatically without human oversight. For security, consider setting an approval_func to review and approve code before execution. See the CodeExecutorAgent documentation for examples of approval functions.
  code_executor = CodeExecutorAgent(


In [51]:
# ============================================================
# 5.1 EXECUTOR WRITE AND CSV LOAD TEST
# ============================================================

# VS Code notebooks use a selector loop on Windows. Run the local executor in
# a worker thread with a Proactor loop so subprocess execution is supported.
import asyncio
from pathlib import Path
from autogen_core import CancellationToken
from autogen_core.code_executor import CodeBlock


def run_executor_blocks(code):
    def worker():
        if os.name == "nt":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

        async def execute_in_worker():
            await executor.start()
            return await executor.execute_code_blocks(
                [CodeBlock(language="python", code=code)],
                cancellation_token=CancellationToken(),
            )

        return asyncio.run(execute_in_worker())

    return worker()


test_code = f"""
from pathlib import Path
import pandas as pd

output_dir = Path({output_folder!r})
data_path = Path({DATA_FILE!r})
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "test.txt").write_text(
    "executor write test passed",
    encoding="utf-8",
)
df = pd.read_csv(data_path)
print(f"Loaded {{data_path}}: {{df.shape[0]}} rows x {{df.shape[1]}} columns")
print(f"Created {{output_dir / 'test.txt'}}")
"""

executor_test_result = await asyncio.to_thread(
    run_executor_blocks,
    test_code,
)

print(executor_test_result)

if executor_test_result.exit_code != 0:
    raise RuntimeError(executor_test_result.output)

assert os.path.isfile(os.path.join(output_folder, "test.txt"))

CommandLineCodeResult(exit_code=0, output='Loaded c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\data\\bond_portfolio_data.csv: 300 rows x 42 columns\r\nCreated c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\test.txt\r\n', code_file='C:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\coding\\tmp_code_31a436e82073989bbd54a1a5d06a5bbc124acbe31993e442b41fc7b3e372539e.py')


In [41]:
# ============================================================
# DATA LOADER AGENT
# ============================================================

data_loader = AssistantAgent(
    name="Data_Loader",
    model_client=model_client,

    system_message="""
You are the Data Loader agent.

Your job is to load and inspect datasets.

Responsibilities:

1. Locate the CSV file provided in the task.
2. Load it using pandas.
3. Display:
   - Number of rows
   - Number of columns
   - Column names
   - Data types
   - First few rows
4. Check whether the file loaded successfully.
5. Identify potential data-quality problems.
6. Do not modify the original dataset.

Use Python code whenever necessary.

Always clearly communicate your findings to the next agent.
"""
)

print("Data_Loader created.")

Data_Loader created.


In [42]:
# ============================================================
# DATA CLEANER AGENT
# ============================================================

data_cleaner = AssistantAgent(
    name="Data_Cleaner",
    model_client=model_client,

    system_message="""
You are the Data Cleaner agent.

Your job is to clean the dataset provided by the Data Loader.

Responsibilities:

1. Load the dataset.
2. Check missing values.
3. Check duplicate rows.
4. Check incorrect or suspicious data types.
5. Handle missing values appropriately.
6. Remove duplicate rows where appropriate.
7. Preserve meaningful information.
8. Do not modify the original CSV.

Save the cleaned dataset as:

outputs/cleaned_data.csv

Print:
- Original row count
- Number of missing values
- Number of duplicates
- Final row count
- Cleaning actions performed

Use Python and pandas.

Do not invent values.
"""
)

print("Data_Cleaner created.")

Data_Cleaner created.


In [43]:
# ============================================================
# FEATURE ENGINEERING AGENT
# ============================================================

feature_engineer = AssistantAgent(
    name="Feature_Engineer",
    model_client=model_client,

    system_message="""
You are the Feature Engineering agent.

Your job is to create useful features from the cleaned dataset.

Responsibilities:

1. Load:
   outputs/cleaned_data.csv

2. Inspect all available columns.

3. Identify useful numerical or derived features.

4. Create features only when they provide analytical value.

5. Avoid unnecessary or meaningless features.

6. Explain every feature that you create.

7. Save the final dataset as:

outputs/feature_engineered_data.csv

Print:
- Original columns
- New columns
- Explanation of new features

Use Python and pandas.

Do not invent data.
"""
)

print("Feature_Engineer created.")

Feature_Engineer created.


In [44]:
# ============================================================
# DATA ANALYZER AGENT
# ============================================================

data_analyzer = AssistantAgent(
    name="Data_Analyzer",
    model_client=model_client,

    system_message="""
You are the Data Analyzer agent.

Your job is to perform statistical and exploratory analysis.

Load:

outputs/feature_engineered_data.csv

Perform:

1. Descriptive statistics
2. Numerical analysis
3. Categorical analysis where applicable
4. Important relationships between variables
5. Correlations where meaningful
6. Identification of unusual observations
7. Important trends

Report factual findings supported by the dataset.

Do not invent statistics.

Print important results clearly so that the Report Generator
can use them later.
"""
)

print("Data_Analyzer created.")

Data_Analyzer created.


In [45]:
# ============================================================
# VISUALIZER AGENT
# ============================================================

visualizer = AssistantAgent(
    name="Visualizer",
    model_client=model_client,

    system_message="""
You are the Visualization agent.

Your job is to create useful visualizations from:

outputs/feature_engineered_data.csv

Use:

- pandas
- matplotlib

Create only charts that are useful for understanding the dataset.

Possible visualizations include:

- Distribution plots
- Bar charts
- Line charts
- Scatter plots
- Correlation heatmaps

Every chart must have:

- Meaningful title
- X-axis label
- Y-axis label
- Readable formatting

Save all charts inside:

outputs/

Use descriptive filenames.

At the end, report the names of all generated files.

Do not invent data.
"""
)

print("Visualizer created.")

Visualizer created.


In [46]:
# ============================================================
# REPORT GENERATOR AGENT
# ============================================================

report_generator = AssistantAgent(
    name="Report_Generator",
    model_client=model_client,

    system_message="""
You are the Report Generator agent.

Your job is to create the final data analysis report.

Use the findings produced by the previous agents.

The report must contain:

# Data Analysis Report

## 1. Dataset Overview

Include:
- Dataset name
- Number of rows
- Number of columns
- Important columns

## 2. Data Cleaning

Include:
- Missing values
- Duplicate rows
- Cleaning operations

## 3. Feature Engineering

Include:
- Features created
- Why they were created

## 4. Statistical Analysis

Include:
- Important statistics
- Important relationships

## 5. Key Insights

List the most important factual findings.

## 6. Visualizations

List the generated visualization filenames.

## 7. Conclusion

Provide a concise factual conclusion.

Save the report as:

outputs/final_report.md

Do not invent any information.

When everything is successfully completed, respond with:

TERMINATE
"""
)

print("Report_Generator created.")

Report_Generator created.


In [47]:
# ============================================================
# TERMINATION CONDITION
# ============================================================

termination = TextMentionTermination(
    "TERMINATE"
)

print("Termination condition created.")

Termination condition created.


In [48]:
# ============================================================
# MULTI-AGENT TEAM
# ============================================================

team = RoundRobinGroupChat(
    participants=[
        data_loader,
        data_cleaner,
        feature_engineer,
        data_analyzer,
        visualizer,
        report_generator,
        code_executor,
    ],

    termination_condition=termination,
)

print("Multi-agent team created successfully.")

Multi-agent team created successfully.


In [49]:
# ============================================================
# ANALYSIS TASK
# ============================================================

task_prompt = f"""
Use the Data_Loader agent to load and inspect this exact CSV file:

{DATA_FILE}

The file must be loaded with pandas before any other analysis begins.
Do not guess a filename and do not use data/input.csv.

Follow this workflow:

1. Data_Loader
   - Load the exact CSV path above.
   - Report rows, columns, data types, missing values, duplicates, and sample rows.

2. Data_Cleaner
   - Clean the loaded dataframe without modifying the original CSV.
   - Save the cleaned data to the absolute path:
     {os.path.join(output_folder, "cleaned_data.csv")}

3. Feature_Engineer
   - Load the cleaned dataset created by Data_Cleaner.
   - Create useful derived features.
   - Save the result to the absolute path:
     {os.path.join(output_folder, "feature_engineered_data.csv")}

4. Data_Analyzer
   - Perform statistical analysis using the feature-engineered dataset.
   - Report only factual findings from the loaded data.

5. Visualizer
   - Create useful matplotlib visualizations.
   - Save PNG files inside:
     {output_folder}

6. Report_Generator
   - Create the final report at the absolute path:
     {os.path.join(output_folder, "final_report.md")}

Rules:
- Use the exact CSV path provided above.
- Use pandas for CSV processing.
- Use matplotlib for visualization.
- Create output directories if necessary.
- Do not modify the original dataset.
- Do not invent data or statistics.
- Print important results.
- Verify that every expected output exists before finishing.

When the entire pipeline is complete, respond with TERMINATE.
"""

print(task_prompt)


Use the Data_Loader agent to load and inspect this exact CSV file:

c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\data\bond_portfolio_data.csv

The file must be loaded with pandas before any other analysis begins.
Do not guess a filename and do not use data/input.csv.

Follow this workflow:

1. Data_Loader
   - Load the exact CSV path above.
   - Report rows, columns, data types, missing values, duplicates, and sample rows.

2. Data_Cleaner
   - Clean the loaded dataframe without modifying the original CSV.
   - Save the cleaned data to the absolute path:
     c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\cleaned_data.csv

3. Feature_Engineer
   - Load the cleaned dataset created by Data_Cleaner.
   - Create useful derived features.
   - Save the result to the absolute path:
     c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\feature_engineered_data.csv

4. Data_Analyzer
   - Perform statistical analysis using the feature-eng

In [52]:
# ============================================================
# 13. RUN ANALYSIS  (improved version)
# ============================================================
# Drop-in replacement for the `pipeline_code` cell. Same structure and
# same downstream call (`run_executor_blocks(pipeline_code)`), so you
# can paste this whole block in place of the old `pipeline_code = r'''..."""`
# without touching anything else in the notebook.

pipeline_code = r'''
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_PATH = Path('../../data/bond_portfolio_data.csv')
OUTPUT_DIR = Path('../../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Global chart styling -> makes every figure look consistent/professional
# ------------------------------------------------------------------
PALETTE = ["#1f4e79", "#2f6f9f", "#4c956c", "#bc4749", "#e8a03d", "#6a4c93", "#8d99ae"]
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#4a4a4a",
    "axes.labelcolor": "#2b2b2b",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.grid": True,
    "grid.color": "#e3e3e3",
    "grid.linewidth": 0.7,
    "xtick.color": "#2b2b2b",
    "ytick.color": "#2b2b2b",
    "font.size": 10,
    "font.family": "DejaVu Sans",
    "figure.dpi": 160,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
})

source_df = pd.read_csv(DATA_PATH)
original_df = source_df.copy(deep=True)
original_rows, original_columns = source_df.shape
missing_before = source_df.isna().sum()
missing_percent = (missing_before / original_rows * 100).round(2)
duplicates_before = int(source_df.duplicated().sum())

# Data cleaning
cleaned_df = source_df.drop_duplicates().copy()
for column in cleaned_df.select_dtypes(include='number').columns:
    if cleaned_df[column].isna().any():
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].median())
for column in cleaned_df.select_dtypes(exclude='number').columns:
    if cleaned_df[column].isna().any():
        cleaned_df[column] = cleaned_df[column].fillna('Unknown')
cleaned_path = OUTPUT_DIR / 'cleaned_data.csv'
cleaned_df.to_csv(cleaned_path, index=False)

# Feature engineering
feature_df = cleaned_df.copy()
created_features = []
if {'MarketValue_INR', 'Quantity'}.issubset(feature_df.columns) and 'MarketValuePerUnit' not in original_df.columns:
    feature_df['MarketValuePerUnit'] = feature_df['MarketValue_INR'] / feature_df['Quantity'].replace(0, pd.NA)
    created_features.append(('MarketValuePerUnit', 'MarketValue_INR / Quantity', 'Market value per unit.'))
if {'YieldToMaturity', 'CouponRate'}.issubset(feature_df.columns) and 'YieldMinusCoupon' not in original_df.columns:
    feature_df['YieldMinusCoupon'] = feature_df['YieldToMaturity'] - feature_df['CouponRate']
    created_features.append(('YieldMinusCoupon', 'YieldToMaturity - CouponRate', 'Difference between yield to maturity and coupon rate.'))
feature_path = OUTPUT_DIR / 'feature_engineered_data.csv'
feature_df.to_csv(feature_path, index=False)

# ------------------------------------------------------------------
# Helpers for report tables
# ------------------------------------------------------------------
def md_table(headers, rows):
    if not rows:
        return '_Not available from the current dataset._'
    header = '| ' + ' | '.join(headers) + ' |'
    divider = '|' + '|'.join(['---'] * len(headers)) + '|'
    body = '\n'.join('| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows)
    return '\n'.join([header, divider, body])

def fmt(value, digits=4):
    if pd.isna(value):
        return 'Not available from the current dataset.'
    return f'{float(value):,.{digits}f}'

def stat_rows(frame, columns):
    rows = []
    for column in columns:
        if column in frame.columns and pd.api.types.is_numeric_dtype(frame[column]):
            values = frame[column].dropna()
            if not values.empty:
                rows.append((column, fmt(values.mean()), fmt(values.median()), fmt(values.min()), fmt(values.max())))
    return rows

numeric_columns = source_df.select_dtypes(include='number').columns.tolist()
date_columns = []
for column in source_df.columns:
    if 'date' in column.lower():
        parsed = pd.to_datetime(source_df[column], errors='coerce', dayfirst=True)
        if parsed.notna().any():
            date_columns.append(column)
categorical_columns = [column for column in source_df.columns if column not in numeric_columns and column not in date_columns]

missing_table = md_table(
    ['Column', 'Missing Values', 'Missing %', 'Data Type'],
    [(column, int(missing_before[column]), f'{missing_percent[column]:.2f}%', str(source_df[column].dtype)) for column in source_df.columns],
)
missing_observations = [
    f'- `{column}` contains {int(missing_before[column])} missing values out of {original_rows} records ({missing_percent[column]:.2f}%).'
    for column in source_df.columns if int(missing_before[column]) > 0
]
if not missing_observations:
    missing_observations = ['- No missing values were found in the source dataset.']

changed_types = [
    f'- `{column}`: {original_df[column].dtype} to {cleaned_df[column].dtype}.'
    for column in original_df.columns if original_df[column].dtype != cleaned_df[column].dtype
]
if not changed_types:
    changed_types = ['- No data-type corrections were required.']

feature_rows = [(name, formula, purpose) for name, formula, purpose in created_features]
feature_table = md_table(['Feature', 'Formula / Logic', 'Purpose'], feature_rows)
if not feature_rows:
    feature_table = '_No new features were added because the existing dataset already contained the required derived variables._'

# ------------------------------------------------------------------
# Portfolio composition
# ------------------------------------------------------------------
portfolio_rows = []
total_market_value = None
sector_values = None
if 'MarketValue_INR' in feature_df:
    total_market_value = feature_df['MarketValue_INR'].sum()
    if 'Sector' in feature_df:
        sector_values = feature_df.groupby('Sector')['MarketValue_INR'].sum().sort_values(ascending=False)
        portfolio_rows = [(sector, fmt(value, 2), f'{value / total_market_value * 100:.2f}%') for sector, value in sector_values.items()]
sector_table = md_table(['Sector', 'Market Value (INR)', 'Portfolio Weight %'], portfolio_rows)
portfolio_findings = []
if total_market_value is not None:
    portfolio_findings.append(f'- Total MarketValue_INR: {fmt(total_market_value, 2)}.')
    portfolio_findings.append(f'- Average MarketValue_INR: {fmt(feature_df["MarketValue_INR"].mean(), 2)}.')
    largest_bond = feature_df.loc[feature_df['MarketValue_INR'].idxmax()]
    bond_label = largest_bond.get('BondID', largest_bond.name)
    portfolio_findings.append(f'- Largest bond by market value: {bond_label} ({fmt(largest_bond["MarketValue_INR"], 2)} INR).')
    if 'Issuer' in feature_df:
        issuer_values = feature_df.groupby('Issuer')['MarketValue_INR'].sum().sort_values(ascending=False)
        portfolio_findings.append(f'- Largest issuer by market value: {issuer_values.index[0]} ({fmt(issuer_values.iloc[0], 2)} INR).')
    if sector_values is not None:
        portfolio_findings.append(f'- Largest sector by market value: {sector_values.index[0]} ({fmt(sector_values.iloc[0], 2)} INR).')
else:
    portfolio_findings = ['- MarketValue_INR is not available from the current dataset.']

# Credit rating analysis
rating_rows = []
rating_values = None
if 'CreditRating' in feature_df and 'MarketValue_INR' in feature_df:
    rating_values = feature_df.groupby('CreditRating').agg(Bonds=('CreditRating', 'size'), MarketValue=('MarketValue_INR', 'sum')).sort_values('MarketValue', ascending=False)
    rating_rows = [(rating, int(row.Bonds), fmt(row.MarketValue, 2), f'{row.MarketValue / total_market_value * 100:.2f}%') for rating, row in rating_values.iterrows()]
rating_table = md_table(['Credit Rating', 'Number of Bonds', 'Market Value (INR)', 'Portfolio Weight %'], rating_rows)

# Interest-rate sensitivity and scenario analysis
sensitivity_columns = ['YieldToMaturity', 'ModifiedDuration', 'EffectiveDuration', 'Convexity', 'EffectiveConvexity', 'DV01_Per100Face']
sensitivity_table = md_table(['Metric', 'Mean', 'Median', 'Minimum', 'Maximum'], stat_rows(feature_df, sensitivity_columns))
scenario_columns = [
    ('+50 bps', 'PriceChange_Up50bps'), ('-50 bps', 'PriceChange_Dn50bps'),
    ('+100 bps', 'PriceChange_Up100bps'), ('-100 bps', 'PriceChange_Dn100bps'),
    ('+200 bps', 'PriceChange_Up200bps'), ('-200 bps', 'PriceChange_Dn200bps'),
]
scenario_rows = [(label, fmt(feature_df[column].mean()), fmt(feature_df[column].median()), fmt(feature_df[column].min()), fmt(feature_df[column].max())) for label, column in scenario_columns if column in feature_df]
scenario_table = md_table(['Scenario', 'Mean Price Change', 'Median', 'Minimum', 'Maximum'], scenario_rows)

# Broader statistical analysis and correlations
priority_columns = ['YieldToMaturity', 'CouponRate', 'CleanPrice', 'DirtyPrice', 'ModifiedDuration', 'EffectiveDuration', 'Convexity', 'EffectiveConvexity', 'DV01_Per100Face', 'MarketValue_INR', 'SpreadOverBenchmark_bps', 'OAS_bps', 'ZSpread_bps']
statistics_table = md_table(['Variable', 'Mean', 'Median', 'Min', 'Max'], stat_rows(feature_df, priority_columns))
correlation_candidates = [column for column in sensitivity_columns + ['MarketValue_INR'] if column in feature_df and pd.api.types.is_numeric_dtype(feature_df[column])]
correlation_lines = []
correlation_pairs = []
if len(correlation_candidates) >= 2:
    correlation = feature_df[correlation_candidates].corr()
    for left_index, left in enumerate(correlation_candidates):
        for right in correlation_candidates[left_index + 1:]:
            value = correlation.loc[left, right]
            if pd.notna(value):
                correlation_pairs.append((abs(value), left, right, value))
    for _, left, right, value in sorted(correlation_pairs, reverse=True)[:5]:
        direction = 'positive' if value >= 0 else 'negative'
        correlation_lines.append(f'- `{left}` and `{right}` show a {direction} correlation of {value:.4f}.')
if not correlation_lines:
    correlation_lines = ['- Not available from the current dataset.']

# ------------------------------------------------------------------
# Visualizations (each chart is now styled + saved + embedded in the report)
# ------------------------------------------------------------------
chart_paths = []
def save_chart(filename, title, xlabel, ylabel):
    ax = plt.gca()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path)
    plt.close()
    chart_paths.append(path)

# 1) Yield distribution
if 'YieldToMaturity' in feature_df:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(feature_df['YieldToMaturity'].dropna(), bins=20, color=PALETTE[1], edgecolor='white', linewidth=0.6)
    mean_yield = feature_df['YieldToMaturity'].mean()
    ax.axvline(mean_yield, color=PALETTE[3], linestyle='--', linewidth=1.4, label=f'Mean = {mean_yield:.2f}')
    ax.legend(frameon=False)
    save_chart('yield_distribution.png', 'Yield to Maturity Distribution', 'Yield to Maturity', 'Number of Bonds')

# 2) Market value by sector
if sector_values is not None:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    bars = ax.bar(sector_values.index.astype(str), sector_values.values, color=PALETTE[2])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    for bar, value in zip(bars, sector_values.values):
        ax.annotate(f'{value:,.0f}', (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 3), textcoords='offset points')
    plt.xticks(rotation=35, ha='right')
    save_chart('market_value_by_sector.png', 'Market Value by Sector', 'Sector', 'Market Value (INR)')

# 3) Credit rating breakdown (new: makes use of the rating table already computed)
if rating_values is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(rating_values.index.astype(str), rating_values['MarketValue'].values, color=PALETTE[4])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    for bar, value in zip(bars, rating_values['MarketValue'].values):
        ax.annotate(f'{value:,.0f}', (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 3), textcoords='offset points')
    save_chart('market_value_by_rating.png', 'Market Value by Credit Rating', 'Credit Rating', 'Market Value (INR)')

# 4) Duration vs yield
if {'ModifiedDuration', 'YieldToMaturity'}.issubset(feature_df.columns):
    fig, ax = plt.subplots(figsize=(9, 5))
    color_values = feature_df['MarketValue_INR'] if 'MarketValue_INR' in feature_df else None
    scatter = ax.scatter(feature_df['ModifiedDuration'], feature_df['YieldToMaturity'], alpha=0.75,
                          c=color_values if color_values is not None else PALETTE[3],
                          cmap='viridis' if color_values is not None else None,
                          edgecolor='white', linewidth=0.4, s=45)
    if color_values is not None:
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Market Value (INR)')
    save_chart('duration_vs_yield.png', 'Modified Duration vs. Yield to Maturity', 'Modified Duration', 'Yield to Maturity')

# 5) Correlation heatmap
if len(correlation_candidates) >= 2:
    fig, ax = plt.subplots(figsize=(9, 7))
    correlation = feature_df[correlation_candidates].corr()
    image = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
    cbar = plt.colorbar(image, ax=ax, label='Correlation')
    ax.set_xticks(range(len(correlation_candidates)))
    ax.set_xticklabels(correlation_candidates, rotation=45, ha='right')
    ax.set_yticks(range(len(correlation_candidates)))
    ax.set_yticklabels(correlation_candidates)
    for row in range(len(correlation_candidates)):
        for column in range(len(correlation_candidates)):
            value = correlation.iloc[row, column]
            text_color = 'white' if abs(value) > 0.6 else '#2b2b2b'
            ax.text(column, row, f'{value:.2f}', ha='center', va='center', fontsize=8, color=text_color)
    ax.grid(False)
    save_chart('correlation_heatmap.png', 'Selected Numeric Feature Correlations', '', '')

if not chart_paths:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.text(0.5, 0.5, 'No chartable columns available', ha='center', va='center')
    ax.axis('off')
    save_chart('analysis.png', 'No Chartable Data', '', '')

# ------------------------------------------------------------------
# Report sections
# ------------------------------------------------------------------
report_path = OUTPUT_DIR / 'final_report.md'
chart_meta = {
    'yield_distribution.png': ('Yield to Maturity Distribution', 'Distribution of observed YieldToMaturity values across the portfolio.'),
    'market_value_by_sector.png': ('Market Value by Sector', 'Total MarketValue_INR grouped by sector.'),
    'market_value_by_rating.png': ('Market Value by Credit Rating', 'Total MarketValue_INR grouped by credit rating.'),
    'duration_vs_yield.png': ('Modified Duration vs. Yield', 'Relationship between ModifiedDuration and YieldToMaturity, colored by market value where available.'),
    'correlation_heatmap.png': ('Correlation Heatmap', 'Pairwise correlations among selected numerical variables.'),
    'analysis.png': ('Fallback Chart', 'Generated when no chartable columns were available.'),
}
chart_sections = []
toc_chart_links = []
for index, path in enumerate(chart_paths, start=1):
    title, purpose = chart_meta.get(path.name, (path.stem.replace('_', ' ').title(), 'Generated from the actual dataset.'))
    anchor = f'13{index}-' + title.lower().replace(' ', '-').replace('.', '')
    toc_chart_links.append(f'  - [{title}](#{anchor})')
    chart_sections.append(
        f'### 13.{index} {title}\n\n'
        f'![{title}](outputs/{path.name})\n\n'
        f'**Filename:** `outputs/{path.name}`  \n'
        f'**Purpose:** {purpose}'
    )
chart_sections_text = '\n\n'.join(chart_sections)
toc_chart_text = '\n'.join(toc_chart_links) if toc_chart_links else '  - _Not available from the current dataset._'

columns_text = '\n'.join(f'- {column}' for column in source_df.columns)
existing_output_files = sorted(path for path in OUTPUT_DIR.iterdir() if path.is_file())
generated_files_text = '\n'.join(f'- `outputs/{path.name}`' for path in existing_output_files)
large_missing = '\n'.join(missing_observations)

key_findings = []
key_findings.append(f'1. The dataset contains {original_rows} records and {original_columns} variables.')
if total_market_value is not None:
    key_findings.append(f'2. Total MarketValue_INR is {fmt(total_market_value, 2)}.')
if sector_values is not None:
    key_findings.append(f'3. The largest sector by calculated market value is {sector_values.index[0]}, representing {sector_values.iloc[0] / total_market_value * 100:.2f}% of total market value.')
if 'ModifiedDuration' in feature_df:
    key_findings.append(f'4. ModifiedDuration ranges from {fmt(feature_df["ModifiedDuration"].min())} to {fmt(feature_df["ModifiedDuration"].max())}.')
if 'YieldToMaturity' in feature_df:
    key_findings.append(f'5. YieldToMaturity ranges from {fmt(feature_df["YieldToMaturity"].min())} to {fmt(feature_df["YieldToMaturity"].max())}.')
key_findings.extend([f'{index + len(key_findings) + 1}. {line[2:]}' for index, line in enumerate(correlation_lines[:3]) if not line.startswith('- Not available')])

# KPI strip for the executive summary (badge-style markdown table)
kpi_cells = [
    ('Records', f'{original_rows:,}'),
    ('Variables', f'{original_columns}'),
    ('Missing Values', f'{int(missing_before.sum()):,}'),
    ('Duplicate Rows', f'{duplicates_before:,}'),
]
if total_market_value is not None:
    kpi_cells.append(('Total Market Value (INR)', fmt(total_market_value, 2)))
kpi_header = '| ' + ' | '.join(label for label, _ in kpi_cells) + ' |'
kpi_divider = '|' + '|'.join([':---:'] * len(kpi_cells)) + '|'
kpi_values = '| ' + ' | '.join(f'**{value}**' for _, value in kpi_cells) + ' |'
kpi_table = '\n'.join([kpi_header, kpi_divider, kpi_values])

generated_on = datetime.now().strftime('%d %B %Y, %H:%M')

report = f"""# AI Multi-Agent Bond Portfolio Analysis Report

**Dataset:** `{DATA_PATH.name}`   |   **Generated:** {generated_on}   |   **Pipeline:** AutoGen Multi-Agent Workflow

---

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Dataset Overview](#2-dataset-overview)
3. [Data Quality Analysis](#3-data-quality-analysis)
4. [Data Cleaning](#4-data-cleaning)
5. [Feature Engineering](#5-feature-engineering)
6. [Portfolio Composition](#6-portfolio-composition)
7. [Credit Rating Analysis](#7-credit-rating-analysis)
8. [Interest Rate Sensitivity](#8-interest-rate-sensitivity)
9. [Interest Rate Scenario Analysis](#9-interest-rate-scenario-analysis)
10. [Statistical Analysis](#10-statistical-analysis)
11. [Correlation Analysis](#11-correlation-analysis)
12. [Key Findings](#12-key-findings)
13. [Visualizations](#13-visualizations)
{toc_chart_text}
14. [Multi-Agent Pipeline](#14-multi-agent-pipeline)
15. [Generated Files](#15-generated-files)
16. [Conclusion](#16-conclusion)

---

## 1. Executive Summary

This report presents a data-grounded analysis of `{DATA_PATH.name}` using an AutoGen multi-agent workflow. The source contains {original_rows} records and {original_columns} variables, with {int(missing_before.sum())} missing values across {int((missing_before > 0).sum())} columns and {duplicates_before} duplicate rows identified prior to cleaning. Portfolio composition, interest-rate sensitivity, scenario fields, correlations, and generated visualizations are reported only when available in the actual dataset.

{kpi_table}

---

## 2. Dataset Overview

| Metric | Value |
|---|---:|
| Dataset | {DATA_PATH.name} |
| Rows | {original_rows} |
| Columns | {original_columns} |
| Numerical Columns | {len(numeric_columns)} |
| Categorical Columns | {len(categorical_columns)} |
| Date Columns | {len(date_columns)} |

### Columns

{columns_text}

---

## 3. Data Quality Analysis

### Missing Values and Data Types

{missing_table}

### Observations

{large_missing}

- Duplicate rows detected: {duplicates_before}.

---

## 4. Data Cleaning

| Metric | Before | After |
|---|---:|---:|
| Rows | {original_rows} | {len(cleaned_df)} |
| Columns | {original_columns} | {len(cleaned_df.columns)} |

- Duplicate rows removed: {duplicates_before}.
- Numeric missing values were imputed with each column's median when missing values existed.
- Non-numeric missing values were replaced with `Unknown` when missing values existed.
- Data-type corrections:
{chr(10).join(changed_types)}
- The original CSV was preserved and not overwritten.

---

## 5. Feature Engineering

{feature_table}

The feature-engineered dataset was saved to `outputs/feature_engineered_data.csv`.

---

## 6. Portfolio Composition

### Portfolio Findings

{chr(10).join(portfolio_findings)}

### Sector Composition

{sector_table}

---

## 7. Credit Rating Analysis

{rating_table}

---

## 8. Interest Rate Sensitivity

The following statistics describe the available yield, duration, convexity, and DV01 fields. They describe the observed data and do not assign a subjective risk classification.

{sensitivity_table}

---

## 9. Interest Rate Scenario Analysis

Scenario values are reported in the units stored in the source columns.

{scenario_table}

---

## 10. Statistical Analysis

{statistics_table}

---

## 11. Correlation Analysis

Correlation describes association and does not establish causation.

{chr(10).join(correlation_lines)}

---

## 12. Key Findings

{chr(10).join(key_findings[:8])}

All findings are derived from calculated values in the source dataset.

---

## 13. Visualizations

{chart_sections_text}

---

## 14. Multi-Agent Pipeline

- **Data Loader:** Loads the source CSV and inspects its structure and quality.
- **Data Cleaner:** Removes duplicate rows and handles missing values without changing the original CSV.
- **Feature Engineer:** Adds only genuinely new derived columns.
- **Data Analyzer:** Calculates descriptive statistics, correlations, and factual findings.
- **Visualizer:** Creates styled charts from available dataset columns.
- **Report Generator:** Validates and writes this report from actual outputs.

```text
Data Loader
     |
Data Cleaner
     |
Feature Engineer
     |
Data Analyzer
     |
Visualizer
     |
Report Generator
```

---

## 15. Generated Files

{generated_files_text}

---

## 16. Conclusion

The analysis covered {original_rows} records and {original_columns} variables. Data quality review found {int(missing_before.sum())} missing values and {duplicates_before} duplicate rows in the original dataset. Portfolio composition was calculated from the available market-value and grouping columns. Interest-rate sensitivity and scenario statistics were calculated only for fields present in the source. The original CSV was not modified, and all listed report artifacts were validated before saving.

---

*Report generated automatically by the AI Multi-Agent Data Analysis pipeline on {generated_on}.*
"""

# Final validation before writing the report
assert original_rows == len(source_df)
assert original_columns == len(source_df.columns)
assert list(source_df.columns) == list(original_df.columns)
assert duplicates_before == int(original_df.duplicated().sum())
assert set(feature_df.columns) - set(original_df.columns) == {item[0] for item in created_features}
assert all(path.is_file() for path in chart_paths)
report_path.write_text(report, encoding='utf-8')
assert report_path.is_file()
print('Final report saved to outputs/final_report.md')

required_paths = [cleaned_path, feature_path, report_path] + chart_paths
for required_path in required_paths:
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
    print(f'Created: {required_path}')
'''

pipeline_result = await asyncio.to_thread(run_executor_blocks, pipeline_code)
print(pipeline_result)
if getattr(pipeline_result, 'exit_code', 0) != 0:
    absolute_data_path = Path(DATA_FILE).resolve()
    absolute_output_dir = Path(output_folder).resolve()

    pipeline_code = pipeline_code.replace(
        "DATA_PATH = Path('../../data/bond_portfolio_data.csv')",
        f"DATA_PATH = Path({str(absolute_data_path)!r})",
    ).replace(
        "OUTPUT_DIR = Path('../../outputs')",
        f"OUTPUT_DIR = Path({str(absolute_output_dir)!r})",
    )

    pipeline_result = await asyncio.to_thread(run_executor_blocks, pipeline_code)
    print(pipeline_result)

    if getattr(pipeline_result, "exit_code", 0) != 0:
        raise RuntimeError(pipeline_result)

CommandLineCodeResult(exit_code=1, output='Traceback (most recent call last):\r\n  File "C:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\coding\\tmp_code_c752439581d1d9a2ba53f05e583872ffe2283ffac4b13b5d46fb4b1de76cb37e.py", line 36, in <module>\r\n    source_df = pd.read_csv(DATA_PATH)\r\n                ^^^^^^^^^^^^^^^^^^^^^^\r\n  File "c:\\Users\\ADITHYA UBALE\\miniconda3\\envs\\AutoGen\\Lib\\site-packages\\pandas\\io\\parsers\\readers.py", line 872, in read_csv\r\n    return _read(filepath_or_buffer, kwds)\r\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\r\n  File "c:\\Users\\ADITHYA UBALE\\miniconda3\\envs\\AutoGen\\Lib\\site-packages\\pandas\\io\\parsers\\readers.py", line 300, in _read\r\n    parser = TextFileReader(filepath_or_buffer, **kwds)\r\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\r\n  File "c:\\Users\\ADITHYA UBALE\\miniconda3\\envs\\AutoGen\\Lib\\site-packages\\pandas\\io\\parsers\\readers.py", line 1643, in __init__\r\n    self._engine = se

In [ ]:
# ============================================================
# 13.1 REPORT GENERATION RETRY WITH RESOLVED PATHS
# ============================================================

# The executor may use either coding/ or research/coding/ as its working
# directory. Inject the notebook-resolved paths so report generation is
# independent of that working-directory detail.
resolved_pipeline_code = pipeline_code.replace(
    "DATA_PATH = Path('../../data/bond_portfolio_data.csv')",
    f"DATA_PATH = Path({DATA_FILE!r})",
).replace(
    "OUTPUT_DIR = Path('../../outputs')",
    f"OUTPUT_DIR = Path({output_folder!r})",
)

pipeline_result = await asyncio.to_thread(
    run_executor_blocks,
    resolved_pipeline_code,
)
print(pipeline_result)
if getattr(pipeline_result, 'exit_code', 0) != 0:
    raise RuntimeError(pipeline_result)

CommandLineCodeResult(exit_code=0, output='Final report saved to outputs/final_report.md\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\cleaned_data.csv\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\feature_engineered_data.csv\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\final_report.md\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\yield_distribution.png\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\market_value_by_sector.png\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\duration_vs_yield.png\r\nCreated: c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\outputs\\correlation_heatmap.png\r\n', code_file='C:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\AutoGen_DataAnalyzer\\coding\\tmp_code_2df1974a293b3ab317f3779ae4a2e1f432329

In [ ]:
# ============================================================
# CHECK OUTPUT FILES
# ============================================================

print("\nGenerated files:\n")

for root, dirs, files in os.walk(output_folder):
    for file in files:
        path = os.path.join(root, file)
        print(path)


Generated files:

c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\.gitkeep
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\cleaned_data.csv
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\correlation_heatmap.png
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\duration_vs_yield.png
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\feature_engineered_data.csv
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\final_report.md
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\market_value_by_sector.png
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\test.txt
c:\Users\ADITHYA UBALE\OneDrive\Desktop\AutoGen_DataAnalyzer\outputs\yield_distribution.png


In [ ]:
# ============================================================
# DISPLAY FINAL REPORT
# ============================================================

report_path = os.path.join(output_folder, "final_report.md")

if os.path.exists(report_path):
    with open(
        report_path,
        "r",
        encoding="utf-8"
    ) as file:
        report = file.read()

    print(report)
else:
    print("Final report has not been generated yet.")

# AI Multi-Agent Data Analysis Report

## 1. Executive Summary

This report presents the automated analysis of the bond portfolio dataset using a multi-agent AI data analysis system built with AutoGen and Gemini.

The pipeline performed data loading, cleaning, feature engineering, statistical analysis, visualization, and report generation.

---

## 2. Dataset Overview

| Metric | Value |
|---|---:|
| Dataset | bond_portfolio_data.csv |
| Rows | 300 |
| Columns | 42 |
| Numerical Columns | 30 |
| Categorical Columns | 14 |

### Columns

- BondID
- ISIN
- Issuer
- Sector
- CreditRating
- Currency
- FaceValue
- CouponRate
- CouponFrequency
- BondType
- MaturityDate
- ValuationDate
- YearsToMaturity
- YieldToMaturity
- CleanPrice
- AccruedInterest
- DirtyPrice
- MacaulayDuration
- ModifiedDuration
- Convexity
- DV01_Per100Face
- EffectiveDuration
- EffectiveConvexity
- SpreadOverBenchmark_bps
- OAS_bps
- ZSpread_bps
- KeyRateBucket
- Quantity
- MarketValue_INR
- PortfolioWeight
- IsCallabl

In [ ]:
# ============================================================
# INSPECT CLEANED DATA
# ============================================================

import pandas as pd

cleaned_path = os.path.join(output_folder, "cleaned_data.csv")

if os.path.exists(cleaned_path):
    cleaned_df = pd.read_csv(cleaned_path)

    print("Shape:")
    print(cleaned_df.shape)

    print("\nColumns:")
    print(cleaned_df.columns.tolist())

    print("\nFirst 5 rows:")
    display(cleaned_df.head())
else:
    print("Cleaned dataset not found.")

Shape:
(300, 42)

Columns:
['BondID', 'ISIN', 'Issuer', 'Sector', 'CreditRating', 'Currency', 'FaceValue', 'CouponRate', 'CouponFrequency', 'BondType', 'MaturityDate', 'ValuationDate', 'YearsToMaturity', 'YieldToMaturity', 'CleanPrice', 'AccruedInterest', 'DirtyPrice', 'MacaulayDuration', 'ModifiedDuration', 'Convexity', 'DV01_Per100Face', 'EffectiveDuration', 'EffectiveConvexity', 'SpreadOverBenchmark_bps', 'OAS_bps', 'ZSpread_bps', 'KeyRateBucket', 'Quantity', 'MarketValue_INR', 'PortfolioWeight', 'IsCallable', 'IsPutable', 'IsFloatingRate', 'BenchmarkIndex', 'PriceChange_Up50bps', 'PriceChange_Dn50bps', 'PriceChange_Up100bps', 'PriceChange_Dn100bps', 'PriceChange_Up200bps', 'PriceChange_Dn200bps', 'Portfolio_Duration', 'Portfolio_Convexity']

First 5 rows:


,BondID,ISIN,Issuer,Sector,CreditRating,Currency,FaceValue,CouponRate,CouponFrequency,BondType,...,IsFloatingRate,BenchmarkIndex,PriceChange_Up50bps,PriceChange_Dn50bps,PriceChange_Up100bps,PriceChange_Dn100bps,PriceChange_Up200bps,PriceChange_Dn200bps,Portfolio_Duration,Portfolio_Convexity
0,ZTFI-0001,IN1043321819,Government of India,Government,SOV,INR,100,0.0620,2,Fixed,...,False,IN10YT,0.9207,-0.9098,1.8522,-1.8086,3.7480,-3.5736,0.005709,0.013596
1,ZTFI-0002,IN7940265423,Government of India,Government,SOV,INR,100,0.0773,2,Fixed,...,False,IN10YT,6.1042,-5.6378,12.6747,-10.8091,27.2150,-19.7526,0.007327,0.116413
2,ZTFI-0003,IN7816184959,Government of India,Government,SOV,INR,100,0.0555,2,Fixed,...,False,IN10YT,2.0557,-2.0040,4.1631,-3.9562,8.5332,-7.7056,0.002534,0.012913
3,ZTFI-0004,IN4752553419,Government of India,Government,SOV,INR,100,0.0681,2,Fixed,...,False,IN10YT,6.3440,-5.7961,13.2358,-11.0444,28.6631,-19.8973,0.018936,0.341828
4,ZTFI-0005,IN5030564139,Government of India,Government,SOV,INR,100,0.0750,2,Fixed,...,False,IN10YT,5.4267,-5.0533,11.2267,-9.7333,23.9468,-17.9731,0.065381,0.931694


In [ ]:
# ============================================================
# INSPECT FEATURE-ENGINEERED DATA
# ============================================================

feature_path = os.path.join(output_folder, "feature_engineered_data.csv")

if os.path.exists(feature_path):
    feature_df = pd.read_csv(feature_path)

    print("Shape:")
    print(feature_df.shape)

    print("\nColumns:")
    print(feature_df.columns.tolist())

    print("\nFirst 5 rows:")
    display(feature_df.head())
else:
    print("Feature-engineered dataset not found.")

Shape:
(300, 44)

Columns:
['BondID', 'ISIN', 'Issuer', 'Sector', 'CreditRating', 'Currency', 'FaceValue', 'CouponRate', 'CouponFrequency', 'BondType', 'MaturityDate', 'ValuationDate', 'YearsToMaturity', 'YieldToMaturity', 'CleanPrice', 'AccruedInterest', 'DirtyPrice', 'MacaulayDuration', 'ModifiedDuration', 'Convexity', 'DV01_Per100Face', 'EffectiveDuration', 'EffectiveConvexity', 'SpreadOverBenchmark_bps', 'OAS_bps', 'ZSpread_bps', 'KeyRateBucket', 'Quantity', 'MarketValue_INR', 'PortfolioWeight', 'IsCallable', 'IsPutable', 'IsFloatingRate', 'BenchmarkIndex', 'PriceChange_Up50bps', 'PriceChange_Dn50bps', 'PriceChange_Up100bps', 'PriceChange_Dn100bps', 'PriceChange_Up200bps', 'PriceChange_Dn200bps', 'Portfolio_Duration', 'Portfolio_Convexity', 'MarketValuePerUnit', 'YieldMinusCoupon']

First 5 rows:


,BondID,ISIN,Issuer,Sector,CreditRating,Currency,FaceValue,CouponRate,CouponFrequency,BondType,...,PriceChange_Up50bps,PriceChange_Dn50bps,PriceChange_Up100bps,PriceChange_Dn100bps,PriceChange_Up200bps,PriceChange_Dn200bps,Portfolio_Duration,Portfolio_Convexity,MarketValuePerUnit,YieldMinusCoupon
0,ZTFI-0001,IN1043321819,Government of India,Government,SOV,INR,100,0.0620,2,Fixed,...,0.9207,-0.9098,1.8522,-1.8086,3.7480,-3.5736,0.005709,0.013596,0.990052,0.0054
1,ZTFI-0002,IN7940265423,Government of India,Government,SOV,INR,100,0.0773,2,Fixed,...,6.1042,-5.6378,12.6747,-10.8091,27.2150,-19.7526,0.007327,0.116413,1.091347,-0.0083
2,ZTFI-0003,IN7816184959,Government of India,Government,SOV,INR,100,0.0555,2,Fixed,...,2.0557,-2.0040,4.1631,-3.9562,8.5332,-7.7056,0.002534,0.012913,0.950190,0.0119
3,ZTFI-0004,IN4752553419,Government of India,Government,SOV,INR,100,0.0681,2,Fixed,...,6.3440,-5.7961,13.2358,-11.0444,28.6631,-19.8973,0.018936,0.341828,1.016783,-0.0014
4,ZTFI-0005,IN5030564139,Government of India,Government,SOV,INR,100,0.0750,2,Fixed,...,5.4267,-5.0533,11.2267,-9.7333,23.9468,-17.9731,0.065381,0.931694,1.048627,-0.0048
